In [82]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score

In [83]:
df=pd.read_csv('data.csv')
df.head()

,message_id,message_text,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,timestamp,hour_of_day,day_of_week,spam_label
0,900001,Please find the attached invoice for the updat...,99,11,1,4,0,0,1,56.6,500.0,6.0,2025-11-24 02:00:00,2,0,0
1,900002,Let's catch up tomorrow regarding the timeline...,73,12,0,0,0,0,0,16.6,207.0,0.0,2025-12-17 21:00:00,21,2,0
2,900003,Can you send the report by end of day? next Mo...,67,13,0,0,0,0,1,25.7,418.0,6.0,2025-11-15 13:00:00,13,5,0
3,900004,Can you send the report by end of day? 10:30 A...,64,13,0,4,0,0,1,48.8,276.0,5.0,2025-12-17 23:00:00,23,2,0
4,900005,Could you review the document and share feedba...,84,14,0,0,0,0,1,33.0,683.0,7.0,2025-11-29 11:00:00,11,5,0


- 6.Identify input features and target variable.


In [84]:
df.drop('timestamp', axis=1, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)
input_features = df.columns[2:-1]
target_feature = df.columns[-1]

print("Input Features:")
print(input_features)

print("\nTarget Feature:")
print(target_feature)

Input Features:
Index(['message_length', 'word_count', 'num_urls', 'num_digits',
       'num_special_chars', 'spam_keyword_score', 'legit_keyword_score',
       'sender_activity_score', 'sender_account_age_days',
       'messages_sent_last_24h', 'hour_of_day', 'day_of_week'],
      dtype='object')

Target Feature:
spam_label


- 7.Perform basic data preprocessing (scaling where required).


In [85]:
scaler = StandardScaler()
x=df[input_features]
y=df[target_feature]
x = pd.DataFrame(scaler.fit_transform(x), columns=input_features)

- 8.Split the dataset into training and testing sets.

In [86]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Part C: Baseline Model – K-Nearest Neighbors

- 9.Implement K-Nearest Neighbors (KNN) classifier.

In [87]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train, y_train)
knn_predictions = knn_model.predict(x_test)

- 10.Experiment with different values of K.

In [88]:
neighbors = range(1, 11)

for n in neighbors:
    knn_model = KNeighborsClassifier(n_neighbors=n)
    knn_model.fit(x_train, y_train)
    knn_predictions = knn_model.predict(x_test)
    
    accuracy = accuracy_score(y_test, knn_predictions)
    precision = precision_score(y_test, knn_predictions, average='weighted')
    recall = recall_score(y_test, knn_predictions, average='weighted')
    f1 = f1_score(y_test, knn_predictions, average='weighted')
    
    print(f"KNN (k={n}) - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


KNN (k=1) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=2) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=3) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=4) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=5) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=6) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=7) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=8) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=9) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
KNN (k=10) - Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000


- 11.Analyze how distance metrics affect predictions.

In [91]:

for metric in ['euclidean', 'manhattan']:
    model = KNeighborsClassifier(n_neighbors=5, metric=metric)
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    print(f"Distance Metric: {metric} -> Accuracy: {accuracy_score(y_test, preds):.4f}")

Distance Metric: euclidean -> Accuracy: 1.0000
Distance Metric: manhattan -> Accuracy: 1.0000


- 12.Identify cases where KNN misclassifies messages.

In [93]:

misclassified_idx = y_test[y_test != knn_predictions].index
print("Total misclassified by KNN:", len(misclassified_idx))

if len(misclassified_idx) > 0:
    results = df.loc[misclassified_idx, ['message_text', 'spam_label']].copy()
    results['predicted'] = knn_predictions[y_test != knn_predictions]
    print(results.head())
else:
    print("No misclassifications found on test set.")

Total misclassified by KNN: 0
No misclassifications found on test set.


### Part D: Support Vector Machine Classifier

- 13.Implement Support Vector Machine (SVM) classifier with:

- - Linear kernel

In [89]:
svm_model_linear = SVC(kernel='linear')
svm_model_linear.fit(x_train, y_train)
svm_linear_predictions = svm_model_linear.predict(x_test)

- - RBF or Polynomial kernel

In [90]:
svm_rbf_model = SVC(kernel='rbf')
svm_rbf_model.fit(x_train, y_train)
svm_rbf_predictions = svm_rbf_model.predict(x_test)


- 14.Analyze margin separation and support vectors.


In [94]:
import numpy as np

# Number of support vectors
print("Support vectors per class:", svm_model_linear.n_support_)
print("Total support vectors:", len(svm_model_linear.support_vectors_))

# Margin width = 2 / ||w||
w = svm_model_linear.coef_[0]
margin = 2 / np.linalg.norm(w)
print("Margin width:", margin)

Support vectors per class: [7 5]
Total support vectors: 12
Margin width: 1.922438304199805


- 15.Compare SVM performance with KNN.

In [95]:
print("KNN Accuracy:", accuracy_score(y_test, knn_predictions))
print("SVM (Linear) Accuracy:", accuracy_score(y_test, svm_linear_predictions))
print("SVM (RBF) Accuracy:", accuracy_score(y_test, svm_rbf_predictions))

KNN Accuracy: 1.0
SVM (Linear) Accuracy: 1.0
SVM (RBF) Accuracy: 1.0


### Part E: Naive Bayes Classifier & Probability

- 16.Implement Naive Bayes Classifier.

In [96]:
nb_model = GaussianNB()
nb_model.fit(x_train, y_train)
nb_predictions = nb_model.predict(x_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_predictions))

Naive Bayes Accuracy: 1.0


- 17.Manually compute conditional probabilities for a few sample messages.

In [97]:
# Prior probabilities
p_spam = (df['spam_label'] == 1).mean()
p_ham = (df['spam_label'] == 0).mean()

print(f"P(Spam): {p_spam:.4f}")
print(f"P(Ham): {p_ham:.4f}")

# Conditional probability of having spam keywords
spam_df = df[df['spam_label'] == 1]
ham_df = df[df['spam_label'] == 0]

p_keyword_given_spam = (spam_df['spam_keyword_score'] > 0).mean()
p_keyword_given_ham = (ham_df['spam_keyword_score'] > 0).mean()

print(f"P(spam_keyword > 0 | Spam): {p_keyword_given_spam:.4f}")
print(f"P(spam_keyword > 0 | Ham): {p_keyword_given_ham:.4f}")

P(Spam): 0.1871
P(Ham): 0.8129
P(spam_keyword > 0 | Spam): 0.7657
P(spam_keyword > 0 | Ham): 0.0000


- 18.Demonstrate how Bayes’ Theorem is applied to compute class probabilities.


In [98]:
# Applying Bayes Theorem: P(Spam | Keyword) = (P(Keyword | Spam) * P(Spam)) / P(Keyword)
p_keyword = (p_keyword_given_spam * p_spam) + (p_keyword_given_ham * p_ham)

p_spam_given_keyword = (p_keyword_given_spam * p_spam) / p_keyword
p_ham_given_keyword = (p_keyword_given_ham * p_ham) / p_keyword

print(f"P(Spam | has spam keyword): {p_spam_given_keyword:.4f}")
print(f"P(Ham | has spam keyword): {p_ham_given_keyword:.4f}")

P(Spam | has spam keyword): 1.0000
P(Ham | has spam keyword): 0.0000


- 19.Compare theoretical probability calculations with model predictions.

In [99]:
# Comparing model predicted probabilities for the first test sample
sample = x_test.iloc[[0]]
prob = nb_model.predict_proba(sample)[0]

print("Model Probabilities [P(Ham), P(Spam)]:", prob)
print("Model Prediction:", nb_model.predict(sample)[0])
print("Actual Label:", y_test.iloc[0])

Model Probabilities [P(Ham), P(Spam)]: [0. 1.]
Model Prediction: 1
Actual Label: 1


### Part F: Model Comparison & Evaluation


- 20.Evaluate all models using appropriate classification metrics:
    - Accuracy
    - Precision
    - Recall
    - F1 Score

In [100]:
models = {
    'KNN': knn_predictions,
    'SVM (Linear)': svm_linear_predictions,
    'SVM (RBF)': svm_rbf_predictions,
    'Naive Bayes': nb_predictions
}

for name, preds in models.items():
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
    print(f"Precision: {precision_score(y_test, preds, average='weighted'):.4f}")
    print(f"Recall:    {recall_score(y_test, preds, average='weighted'):.4f}")
    print(f"F1 Score:  {f1_score(y_test, preds, average='weighted'):.4f}\n")

--- KNN ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000

--- SVM (Linear) ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000

--- SVM (RBF) ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000

--- Naive Bayes ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000



- 21.Compare:
    - KNN vs SVM vs Naive Bayes


In [101]:
comparison_df = pd.DataFrame({
    'Model': ['KNN', 'SVM (Linear)', 'SVM (RBF)', 'Naive Bayes'],
    'Accuracy': [accuracy_score(y_test, knn_predictions), accuracy_score(y_test, svm_linear_predictions), accuracy_score(y_test, svm_rbf_predictions), accuracy_score(y_test, nb_predictions)],
    'Precision': [precision_score(y_test, knn_predictions, average='weighted'), precision_score(y_test, svm_linear_predictions, average='weighted'), precision_score(y_test, svm_rbf_predictions, average='weighted'), precision_score(y_test, nb_predictions, average='weighted')],
    'Recall': [recall_score(y_test, knn_predictions, average='weighted'), recall_score(y_test, svm_linear_predictions, average='weighted'), recall_score(y_test, svm_rbf_predictions, average='weighted'), recall_score(y_test, nb_predictions, average='weighted')],
    'F1 Score': [f1_score(y_test, knn_predictions, average='weighted'), f1_score(y_test, svm_linear_predictions, average='weighted'), f1_score(y_test, svm_rbf_predictions, average='weighted'), f1_score(y_test, nb_predictions, average='weighted')]
})
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,KNN,1.0,1.0,1.0,1.0
1,SVM (Linear),1.0,1.0,1.0,1.0
2,SVM (RBF),1.0,1.0,1.0,1.0
3,Naive Bayes,1.0,1.0,1.0,1.0


- 22.Identify which model performs best for:
    - High precision
    - High recall

In [102]:
best_precision = comparison_df.loc[comparison_df['Precision'].idxmax()]
best_recall = comparison_df.loc[comparison_df['Recall'].idxmax()]

print(f"Best model for High Precision: {best_precision['Model']} ({best_precision['Precision']:.4f})")
print(f"Best model for High Recall: {best_recall['Model']} ({best_recall['Recall']:.4f})")

Best model for High Precision: KNN (1.0000)
Best model for High Recall: KNN (1.0000)
